First Step. Choose T4 GPU hardware for training. Then prepare requirement libraries.

In [ ]:
# 1. 安装必要的库 (Colab 环境适配)
!pip install -q transformers peft accelerate bitsandbytes datasets python-dotenv

Second Step. Choose Base Model with small batches.

In [ ]:
"""
轻量级测试版微调脚本
Lightweight Test Version for Quick Verification
"""
import os
import json
from pathlib import Path
import torch
from transformers import (
    LlamaTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from torch.utils.data import Dataset
from peft import get_peft_model, LoraConfig, TaskType
from dotenv import load_dotenv

load_dotenv()

PROJECT_NAME = "be-with-me-kin-test"
DATA_PATH = Path("/content/data")
OUTPUT_DIR = Path("/content/models") / PROJECT_NAME

# 轻量级配置
MODEL_NAME = "ministral/Ministral-3b-instruct"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 2  # 减小批大小
GRAD_ACCUMULATION_STEPS = 2
NUM_EPOCHS = 1  # 只训练 1 个 epoch
MAX_SEQ_LENGTH = 1024  # 减小序列长度
LEARNING_RATE = 1e-3


class SimpleDataset(Dataset):
    """简化后的数据集"""

    def __init__(self, file_path: Path, tokenizer, max_length: int = MAX_SEQ_LENGTH, limit: int = None):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.examples = []

        with open(file_path, 'r', encoding='utf-8') as f:
            for idx, line in enumerate(f):
                if limit and idx >= limit:
                    break

                if not line.strip():
                    continue

                try:
                    entry = json.loads(line)
                    messages = entry.get("messages", [])

                    if not messages or messages[-1]["role"].lower() != "assistant":
                        continue

                    # 格式化对话
                    text = ""
                    for msg in messages:
                        role = msg.get("role", "").upper()
                        content = msg.get("content", "")
                        text += f"[{role}] {content}\n"

                    self.examples.append(text.strip())
                except:
                    continue

        print(f" 加载了 {len(self.examples)} 条数据（限制 {limit}）")

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        text = self.examples[idx]
        encodings = self.tokenizer(
            text,
            max_length=self.max_length,
            truncation=True,
            padding="max_length",
            return_tensors="pt"
        )
        return {
            "input_ids": encodings["input_ids"].squeeze(),
            "attention_mask": encodings["attention_mask"].squeeze(),
            "labels": encodings["input_ids"].squeeze(),
        }


def main_lightweight():
    """轻量级训练"""
    print("""
╔════════════════════════════════════════╗
║  Lightweight Fine-tuning (Test)        ║
║  轻量级微调（快速测试版）                ║
╚════════════════════════════════════════╝
    """)

    print(f" 设备: {DEVICE}")
    print(f" 模型: {MODEL_NAME}")
    print(f" 批大小: {BATCH_SIZE}")
    print(f" Epochs: {NUM_EPOCHS}")

    # 1. 加载分词器
    print("\n 加载分词器...")
    tokenizer = LlamaTokenizer.from_pretrained(
        MODEL_NAME,
        trust_remote_code=True,
        legacy=False
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # 2. 加载数据（限制数量以加快测试）
    print(" 加载数据（仅 100 条用于测试）...")
    train_path = Path(DATA_PATH) / "train_kin.jsonl"
    val_path = Path(DATA_PATH) / "eval_kin.jsonl"

    if not train_path.exists() or not val_path.exists():
        print(" 数据文件不存在！")
        return

    train_dataset = SimpleDataset(train_path, tokenizer, limit=100)
    val_dataset = SimpleDataset(val_path, tokenizer, limit=20)

    import gc
    # 在加载模型前调用
    gc.collect()

    # 3. 加载模型
    print("\n 加载模型（这可能需要几分钟）...")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        device_map={"": 0} if DEVICE == "cuda" else "cpu",
        dtype=torch.bfloat16,
        low_cpu_mem_usage=True,
        trust_remote_code=True,
    )

    # 4. 应用 LoRA
    print(" 应用 LoRA...")
    peft_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        bias="none",
        target_modules=["q_proj", "v_proj"],
    )
    model = get_peft_model(model, peft_config)

    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f" 可训练参数: {trainable_params:,} ({100 * trainable_params / total_params:.1f}%)")

    # 5. 训练
    print("\n 开始训练...")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    training_args = TrainingArguments(
        output_dir=str(OUTPUT_DIR),
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUMULATION_STEPS,
        learning_rate=LEARNING_RATE,
        logging_steps=5,
        eval_strategy="epoch",
        save_strategy="epoch",
        fp16=DEVICE == "cuda",
        gradient_checkpointing=True,
        optim="paged_adamw_32bit",
        report_to=[],
    )

    data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=data_collator,
    )

    train_result = trainer.train()

    # 6. 保存模型
    print("\n 保存模型...")
    trainer.save_model(str(OUTPUT_DIR))
    tokenizer.save_pretrained(str(OUTPUT_DIR))

if __name__ == "__main__":
    main_lightweight()



╔════════════════════════════════════════╗
║  ⚡ Lightweight Fine-tuning (Test)    ║
║  轻量级微调（快速测试版）             ║
╚════════════════════════════════════════╝
    
⚙️  设备: cuda
🤖 模型: ministral/Ministral-3b-instruct
📊 批大小: 2
📈 Epochs: 1

📥 加载分词器...
📖 加载数据（仅 100 条用于测试）...
✅ 加载了 57 条数据（限制 100）
✅ 加载了 12 条数据（限制 20）

📥 加载模型（这可能需要几分钟）...


config.json:   0%|          | 0.00/619 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/129 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

🔧 应用 LoRA...
✅ 可训练参数: 1,490,944 (0.0%)

🚀 开始训练...


Epoch,Training Loss,Validation Loss
1,2.300636,2.249847



💾 保存模型...

✅ 轻量级训练完成！
📁 模型已保存到: /content/models/be-with-me-kin-test
📝 现在可以使用 inference.py 进行推理

下一步:
1. 运行完整微调: python src/training/local_mistral_finetuner.py
2. 进行推理: python src/training/inference.py
    


In [ ]:
import shutil
from google.colab import files

# 定义压缩包名称
zip_name = "lora_weight_kin_test_1"
# 这里的路径要对应你脚本里的输出路径
folder_to_download = "/content/models/be-with-me-kin-test/checkpoint-15"

# 压缩文件夹
shutil.make_archive(zip_name, 'zip', folder_to_download)

# 下载到本地浏览器
files.download(f"{zip_name}.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Formal Stage.Third Step. Choose Base Model with large batches.

In [ ]:
"""
BeWithMe 正式版 SFT 微调脚本
基于原代码结构，针对生产级微调参数进行优化
"""
import os
import json
from pathlib import Path
import torch
import gc
from transformers import (
    AutoTokenizer, # 建议使用 AutoTokenizer 以获得更好的兼容性
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from torch.utils.data import Dataset
from peft import get_peft_model, LoraConfig, TaskType
from dotenv import load_dotenv

load_dotenv()

# --- 路径配置 ---
PROJECT_NAME = "be-with-me-kin-sft"
DATA_PATH = Path("/content/data")
OUTPUT_DIR = Path("/content/models") / PROJECT_NAME

# --- 正式 SFT 参数调整 ---
MODEL_NAME = "ministral/Ministral-3b-instruct"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# 提升 Batch Size 和 Epoch 以保证模型学透数据风格
BATCH_SIZE = 4
GRAD_ACCUMULATION_STEPS = 4    # 等效总 Batch Size = 16
NUM_EPOCHS = 3                # 正式训练通常运行 3 个轮次
MAX_SEQ_LENGTH = 1024
LEARNING_RATE = 2e-4           # SFT 经典微调学习率

class SimpleDataset(Dataset):
    """保持原有的数据集加载逻辑"""
    def __init__(self, file_path: Path, tokenizer, max_length: int = MAX_SEQ_LENGTH, limit: int = None):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.examples = []

        if not file_path.exists():
            return

        with open(file_path, 'r', encoding='utf-8') as f:
            for idx, line in enumerate(f):
                if limit and idx >= limit:
                    break
                if not line.strip():
                    continue

                try:
                    entry = json.loads(line)
                    messages = entry.get("messages", [])

                    if not messages or messages[-1]["role"].lower() != "assistant":
                        continue

                    # 格式化对话：保持原有的文本拼接逻辑
                    text = ""
                    for msg in messages:
                        role = msg.get("role", "").upper()
                        content = msg.get("content", "")
                        text += f"[{role}] {content}\n"

                    self.examples.append(text.strip())
                except:
                    continue

        print(f"✅ 数据集加载完成: {file_path.name} | 条数: {len(self.examples)}")

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        text = self.examples[idx]
        encodings = self.tokenizer(
            text,
            max_length=self.max_length,
            truncation=True,
            padding="max_length",
            return_tensors="pt"
        )
        return {
            "input_ids": encodings["input_ids"].squeeze(),
            "attention_mask": encodings["attention_mask"].squeeze(),
            "labels": encodings["input_ids"].squeeze(),
        }

def main_lightweight():
    """正式 SFT 微调主函数"""
    print(f"\n 开始正式 SFT 微调 | 设备: {DEVICE} | 模型: {MODEL_NAME}")

    # 1. 加载分词器
    print(" 加载分词器...")
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME,
        trust_remote_code=True
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # 2. 检查并加载数据
    train_path = DATA_PATH / "train_kin.jsonl"
    val_path = DATA_PATH / "eval_kin.jsonl"

    if not train_path.exists() or not val_path.exists():
        print(f" 数据文件不存在！请检查路径: {train_path}")
        return

    # 正式微调建议去掉 limit 或设为很大的值
    train_dataset = SimpleDataset(train_path, tokenizer, limit=None)
    val_dataset = SimpleDataset(val_path, tokenizer, limit=None)

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # 3. 加载模型 (开启 4-bit 以适配 Colab 显存并支持更大的参数覆盖)
    print(" 加载模型 (开启 4-bit 量化加速)...")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        device_map="auto",
        torch_dtype=torch.bfloat16,
        low_cpu_mem_usage=True,
        trust_remote_code=True,
    )

    # 4. 优化 LoRA 配置
    print(" 应用 LoRA 配置 (正式版覆盖范围)...")
    peft_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=16,                       # 提升秩，增强角色表现力
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        # 正式 SFT 建议覆盖全量线性层以获得最佳语气模仿效果
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    )
    model = get_peft_model(model, peft_config)
    model.print_trainable_parameters()

    # 5. 调整训练策略
    print(" 开始微调迭代...")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    training_args = TrainingArguments(
        output_dir=str(OUTPUT_DIR),
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUMULATION_STEPS,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type="cosine", # 正式训练使用余弦退火，收敛更平滑
        logging_steps=10,
        eval_strategy="epoch",      # 每个阶段评估一次
        save_strategy="epoch",      # 每个阶段保存一次
        save_total_limit=2,         # 节省 Colab 磁盘空间
        fp16=True,
        gradient_checkpointing=True,
        optim="paged_adamw_32bit",
        report_to=[],
    )

    data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=data_collator,
    )

    trainer.train()

    # 6. 保存最终适配器权重
    print("\n 保存微调后的 LoRA 适配器...")
    final_adapter_path = OUTPUT_DIR / "final_lora_adapter"
    trainer.save_model(str(final_adapter_path))
    tokenizer.save_pretrained(str(final_adapter_path))

    print(f"\n 训练完成！适配器已保存至: {final_adapter_path}")

if __name__ == "__main__":
    main_lightweight()


🚀 开始正式 SFT 微调 | 设备: cuda | 模型: ministral/Ministral-3b-instruct
📥 加载分词器...
✅ 数据集加载完成: train_kin.jsonl | 条数: 285
✅ 数据集加载完成: eval_kin.jsonl | 条数: 30
📥 加载模型 (开启 4-bit 量化加速)...


Loading weights:   0%|          | 0/129 [00:00<?, ?it/s]

🔧 应用 LoRA 配置 (正式版覆盖范围)...
trainable params: 18,350,080 || all params: 3,334,066,176 || trainable%: 0.5504
🚀 开始微调迭代...


Epoch,Training Loss,Validation Loss
1,2.464131,1.950481
2,1.741372,1.866693
3,1.488940,1.861179



💾 保存微调后的 LoRA 适配器...

✅ 训练完成！适配器已保存至: /content/models/be-with-me-kin-sft/final_lora_adapter


In [16]:
# 定义压缩包名称
zip_name = "lora_weight_kin_test_2"
# 这里的路径要对应你脚本里的输出路径
folder_to_download = "/content/models/be-with-me-kin-sft/final_lora_adapter"

# 压缩文件夹
shutil.make_archive(zip_name, 'zip', folder_to_download)

# 下载到本地浏览器
files.download(f"{zip_name}.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>